# Cardiovascular Disease Detection: Complete Tutorial

## Overview
This notebook provides a comprehensive, step-by-step guide to building and comparing three different machine learning approaches for detecting cardiovascular diseases from ECG (electrocardiogram) signals:

1. **Simple Classification Model** - Random Forest using hand-crafted features
2. **CNN (Convolutional Neural Network)** - Deep learning model that learns features automatically
3. **LSTM (Long Short-Term Memory)** - Recurrent neural network for sequential data

### What You'll Learn:
- How to download and prepare medical ECG data
- How to preprocess time-series signals
- How to extract features from signals
- How to build and train three different types of models
- How to evaluate and compare model performance

### Dataset:
We'll use the **PTB-XL dataset**, which contains over 21,000 clinical ECG recordings labeled by cardiologists.

---
## Step 1: Install and Import Required Libraries

First, we need to install all the necessary Python libraries. These libraries help us:
- Process data (numpy, pandas)
- Read ECG files (wfdb)
- Build machine learning models (scikit-learn, tensorflow)
- Visualize results (matplotlib, seaborn)

In [ ]:
# Install required packages
!pip install numpy pandas wfdb scikit-learn tensorflow scipy tqdm matplotlib seaborn pywt

In [ ]:
# Import all necessary libraries
import os
import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt
import seaborn as sns
from ast import literal_eval
import pywt

# Machine Learning libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from scipy.stats import skew, kurtosis

# Deep Learning libraries
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# For progress tracking
from tqdm import tqdm

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---
## Step 2: Download the PTB-XL Dataset

### What is the PTB-XL Dataset?
The PTB-XL is a large database of 21,837 clinical 12-lead ECG recordings. Each recording:
- Is 10 seconds long
- Has been labeled by up to two cardiologists
- Contains diagnosis information for 5 major categories:
  - **NORM**: Normal ECG
  - **MI**: Myocardial Infarction (heart attack)
  - **STTC**: ST/T Change (abnormal heart rhythms)
  - **CD**: Conduction Disturbance (electrical signal problems)
  - **HYP**: Hypertrophy (enlarged heart chambers)

### Why This Dataset?
- Large and diverse (21,000+ recordings)
- Professionally labeled by cardiologists
- Predefined train/validation/test splits (prevents data leakage)
- Publicly available and well-documented

In [ ]:
# Download the PTB-XL dataset
# This will download approximately 900MB of data

import urllib.request
import zipfile

# Create data directory
os.makedirs('data/raw', exist_ok=True)

# URL for the PTB-XL dataset
ptbxl_url = 'https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip'

# Download the dataset
print("Downloading PTB-XL dataset... (this may take several minutes)")
zip_path = 'data/raw/ptb-xl.zip'

# Check if already downloaded
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(ptbxl_url, zip_path)
    print("Download complete!")
else:
    print("Dataset already downloaded.")

# Extract the dataset
print("Extracting dataset...")
if not os.path.exists('data/raw/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('data/raw/')
    print("Extraction complete!")
else:
    print("Dataset already extracted.")

# Set the path to the dataset
data_path = 'data/raw/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
print(f"\nDataset location: {data_path}")

---
## Step 3: Understanding the Dataset Structure

Let's explore the dataset to understand what we're working with.

In [ ]:
# Load the metadata file
database_path = os.path.join(data_path, 'ptbxl_database.csv')
df = pd.read_csv(database_path, index_col='ecg_id')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# Explore the columns
print("Column names:")
print(df.columns.tolist())
print("\nBasic statistics:")
print(df.describe())

---
## Step 4: Data Loading and Preprocessing

### What Happens in This Step:
1. **Load metadata** - Read patient information and diagnoses
2. **Parse diagnostic codes** - Convert cryptic codes to readable labels
3. **Create multi-label targets** - One ECG can have multiple conditions
4. **Load ECG waveforms** - Read the actual signal data
5. **Denoise signals** - Remove noise using wavelet transform
6. **Normalize signals** - Standardize using z-score normalization
7. **Split data** - Separate into train/validation/test sets

### Why Z-Score Normalization?
Different patients have different ECG amplitudes. Normalization ensures all signals are on the same scale:
- **Formula**: `(signal - mean) / std_deviation`
- **Result**: Mean = 0, Standard Deviation = 1
- **Benefit**: Models train faster and more effectively

In [ ]:
def wavelet_denoise(signal, wavelet='db6', level=1):
    """
    Remove noise from ECG signal using wavelet denoising.
    
    What it does:
    - Decomposes signal into different frequency components
    - Removes high-frequency noise
    - Reconstructs clean signal
    
    Args:
        signal: Raw ECG signal
        wavelet: Type of wavelet (db6 is good for ECG)
        level: Decomposition level
    
    Returns:
        Denoised signal
    """
    # Decompose signal using wavelet transform
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    
    # Estimate noise level
    sigma = np.median(np.abs(coeffs[-level])) / 0.6745
    
    # Calculate threshold
    uthresh = sigma * np.sqrt(2 * np.log(len(signal)))
    
    # Apply soft thresholding to remove noise
    coeffs[1:] = [pywt.threshold(i, value=uthresh, mode='soft') for i in coeffs[1:]]
    
    # Reconstruct signal
    return pywt.waverec(coeffs, wavelet)[:len(signal)]

In [ ]:
def load_ptbxl_data(root_dir, sampling_frequency=100):
    """
    Load and preprocess PTB-XL ECG data.
    
    This function performs all the heavy lifting:
    1. Loads metadata (CSV files with patient info)
    2. Processes diagnostic labels
    3. Loads ECG waveforms
    4. Applies preprocessing (denoising, normalization)
    5. Splits data into train/validation/test sets
    
    Args:
        root_dir: Path to PTB-XL dataset
        sampling_frequency: 100 Hz (faster) or 500 Hz (more detailed)
    
    Returns:
        X_train, y_train, X_val, y_val, X_test, y_test, label_columns
    """
    
    print("Step 1: Loading metadata...")
    # Load database metadata
    db_path = os.path.join(root_dir, "ptbxl_database.csv")
    data = pd.read_csv(db_path, index_col='ecg_id')
    print(f"  - Loaded {len(data)} ECG records")
    
    print("\nStep 2: Processing diagnostic labels...")
    # Load SCP statements to map codes to diagnostic superclasses
    scp_path = os.path.join(root_dir, "scp_statements.csv")
    scp_statements = pd.read_csv(scp_path, index_col=0)
    # Only keep diagnostic codes (not symptoms)
    scp_statements = scp_statements[scp_statements.diagnostic == 1]
    
    def aggregate_diagnostic(scp_codes_str):
        """
        Convert cryptic SCP codes to readable diagnostic categories.
        
        Example: {'NORM': 100, 'IMI': 80} -> ['NORM', 'MI']
        """
        scp_codes = literal_eval(scp_codes_str)
        res = set()
        for code, confidence in scp_codes.items():
            if confidence >= 0:  # Check if diagnosis is present
                if code in scp_statements.index:
                    super_class = scp_statements.loc[code].diagnostic_class
                    res.add(super_class)
        return list(res)
    
    # Apply diagnostic aggregation
    data['diagnostic_superclass'] = data.scp_codes.apply(aggregate_diagnostic)
    
    print("\nStep 3: Creating multi-label target vectors...")
    # Get all unique diagnostic categories
    all_superclasses = sorted(list(set(item for sublist in data.diagnostic_superclass for item in sublist)))
    print(f"  - Found {len(all_superclasses)} diagnostic categories: {all_superclasses}")
    
    # Create binary column for each category (multi-label encoding)
    for sc in all_superclasses:
        data[sc] = data.diagnostic_superclass.apply(lambda x: 1 if sc in x else 0)
    
    # Show distribution of classes
    print("\n  - Class distribution:")
    for sc in all_superclasses:
        count = data[sc].sum()
        percentage = (count / len(data)) * 100
        print(f"    {sc}: {count} ({percentage:.1f}%)")
    
    # Extract labels
    label_columns = all_superclasses
    y = data[label_columns].values
    
    print("\nStep 4: Loading ECG waveforms...")
    # Determine which files to use based on sampling frequency
    if sampling_frequency == 100:
        file_col = 'filename_lr'  # Low resolution (100 Hz)
    elif sampling_frequency == 500:
        file_col = 'filename_hr'  # High resolution (500 Hz)
    else:
        raise ValueError("Sampling frequency must be 100 or 500")
    
    # Initialize array to store ECG signals
    # Shape: (number of records, time points, number of leads)
    X = np.zeros((len(data), sampling_frequency * 10, 1))  # 10 seconds, 1 lead
    
    print(f"  - Loading {len(data)} ECG signals at {sampling_frequency} Hz...")
    print("  - Using Lead II (standard lead for heart rhythm analysis)")
    
    # Load each ECG record
    for i, ecg_id in enumerate(tqdm(data.index, desc="  - Progress")):
        file_path = os.path.join(root_dir, data.loc[ecg_id][file_col])
        record = wfdb.rdrecord(file_path)
        
        # Extract Lead II (index 1) - commonly used for rhythm analysis
        signal = record.p_signal[:, 1]
        
        # Apply wavelet denoising
        signal = wavelet_denoise(signal)
        
        # Z-score normalization: (signal - mean) / std
        signal = (signal - np.mean(signal)) / np.std(signal)
        
        X[i, :, 0] = signal
    
    print("\nStep 5: Splitting data into train/validation/test sets...")
    # Use predefined folds to prevent data leakage
    # This ensures no patient appears in multiple sets
    train_fold = data[data.strat_fold <= 8]
    val_fold = data[data.strat_fold == 9]
    test_fold = data[data.strat_fold == 10]
    
    X_train = X[train_fold.index.map(data.index.get_loc)]
    y_train = y[train_fold.index.map(data.index.get_loc)]
    
    X_val = X[val_fold.index.map(data.index.get_loc)]
    y_val = y[val_fold.index.map(data.index.get_loc)]
    
    X_test = X[test_fold.index.map(data.index.get_loc)]
    y_test = y[test_fold.index.map(data.index.get_loc)]
    
    print("\n" + "="*60)
    print("DATA LOADING COMPLETE!")
    print("="*60)
    print(f"Label columns: {label_columns}")
    print(f"\nX_train shape: {X_train.shape} - {X_train.shape[0]} samples")
    print(f"y_train shape: {y_train.shape}")
    print(f"\nX_val shape:   {X_val.shape} - {X_val.shape[0]} samples")
    print(f"y_val shape:   {y_val.shape}")
    print(f"\nX_test shape:  {X_test.shape} - {X_test.shape[0]} samples")
    print(f"y_test shape:  {y_test.shape}")
    print("="*60)
    
    return X_train, y_train, X_val, y_val, X_test, y_test, label_columns

In [ ]:
# Load the data (this will take several minutes)
X_train, y_train, X_val, y_val, X_test, y_test, label_columns = load_ptbxl_data(
    root_dir=data_path,
    sampling_frequency=100  # Use 100 Hz for faster processing
)

num_classes = y_train.shape[1]
print(f"\nNumber of classes: {num_classes}")

---
## Step 5: Visualize Sample ECG Signals

Let's visualize some ECG signals to understand what we're working with.

In [ ]:
# Plot sample ECG signals from each class
fig, axes = plt.subplots(num_classes, 1, figsize=(15, 12))
fig.suptitle('Sample ECG Signals for Each Diagnostic Class', fontsize=16, y=1.00)

for i, label in enumerate(label_columns):
    # Find first sample with this label
    idx = np.where(y_train[:, i] == 1)[0][0]
    
    # Plot the signal
    time = np.arange(len(X_train[idx])) / 100  # Convert to seconds
    axes[i].plot(time, X_train[idx, :, 0], linewidth=0.8)
    axes[i].set_title(f'{label} - Sample ECG', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Time (seconds)')
    axes[i].set_ylabel('Amplitude (normalized)')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nWhat you're seeing:")
print("- Each plot shows 10 seconds of ECG signal")
print("- The y-axis is normalized (z-score), so values are centered around 0")
print("- Different patterns indicate different heart conditions")
print("- Normal rhythms are more regular, abnormal ones show irregularities")

---
## Step 6: Feature Extraction for Simple Classification Model

### What Are Features?
Features are measurable properties of the signal that help distinguish between different classes. For ECG signals, we extract:

#### Time-Domain Features:
These describe the shape and amplitude of the signal:
- **Mean & Standard Deviation**: Overall signal characteristics
- **Maximum & Minimum**: Peak amplitudes
- **Skewness**: Asymmetry of the signal
- **Kurtosis**: "Peakedness" of the distribution
- **QRS Width**: Duration of the heart's electrical activation
- **Zero Crossing Rate**: How often signal crosses zero

#### Frequency-Domain Features:
These describe the rhythmic components:
- **Band Energy**: Power in different frequency ranges
- **Spectral Entropy**: Randomness in frequency distribution
- **Dominant Frequency**: Main rhythmic component
- **Spectral Centroid**: Center of mass of spectrum

### Why Extract Features?
Traditional machine learning models (like Random Forest) can't work with raw signals directly. They need numerical features that capture the important characteristics of the data.

In [ ]:
def extract_time_features(window, fs=100):
    """
    Extract time-domain features from ECG signal.
    
    These features describe the shape, amplitude, and temporal
    characteristics of the signal.
    """
    feat = []
    
    # Basic statistics
    feat.append(np.mean(window))           # Average amplitude
    feat.append(np.std(window))            # Signal variability
    feat.append(np.max(window))            # Peak amplitude (R wave)
    feat.append(np.min(window))            # Minimum amplitude (S wave)
    feat.append(skew(window))              # Asymmetry of distribution
    feat.append(kurtosis(window))          # Peakedness of distribution
    
    # Peak-to-peak amplitude (QRS complex amplitude)
    feat.append(np.max(window) - np.min(window))
    
    # QRS width estimate (important for arrhythmia detection)
    r_idx = np.argmax(window)  # Find R peak
    qrs_region = window[max(0, r_idx - 50):min(len(window), r_idx + 50)]
    half_peak = 0.5 * window[r_idx]
    qrs_width = np.sum(qrs_region > half_peak) / fs
    feat.append(qrs_width)
    
    # Area under curve (total electrical activity)
    feat.append(np.sum(np.abs(window)) / fs)
    
    # Mean absolute deviation
    feat.append(np.mean(np.abs(window - np.mean(window))))
    
    # Root mean square (signal energy)
    feat.append(np.sqrt(np.mean(window ** 2)))
    
    # Zero crossing rate (rhythm regularity)
    feat.append(np.sum(np.diff(np.signbit(window))) / len(window))
    
    # Signal energy
    feat.append(np.sum(window ** 2) / len(window))
    
    return np.array(feat)


def extract_frequency_features(window, fs=100):
    """
    Extract frequency-domain features using Fourier Transform.
    
    FFT decomposes the signal into different frequency components,
    helping identify rhythmic abnormalities.
    """
    # Compute Fast Fourier Transform (FFT)
    fft_vals = np.abs(np.fft.rfft(window))
    freqs = np.fft.rfftfreq(len(window), 1/fs)
    
    # Define frequency bands (in Hz)
    # These bands capture different physiological rhythms
    bands = {
        'delta': (0, 4),      # Very slow rhythms
        'theta': (4, 8),      # Slow rhythms
        'alpha': (8, 13),     # Medium rhythms
        'beta': (13, 30),     # Fast rhythms
        'gamma': (30, 100)    # Very fast components
    }
    
    feat = []
    
    # Energy in each frequency band
    for band_name, (low, high) in bands.items():
        mask = (freqs >= low) & (freqs < high)
        band_energy = np.sum(fft_vals[mask] ** 2)
        feat.append(band_energy)
    
    # Spectral entropy (measure of frequency complexity)
    psd = fft_vals ** 2
    psd = psd / np.sum(psd)
    spectral_entropy = -np.sum(psd * np.log2(psd + 1e-10))
    feat.append(spectral_entropy)
    
    # Dominant frequency (main rhythmic component)
    dominant_freq = freqs[np.argmax(fft_vals)]
    feat.append(dominant_freq)
    
    # Spectral centroid (center of mass of spectrum)
    centroid = np.sum(freqs * fft_vals) / np.sum(fft_vals)
    feat.append(centroid)
    
    return np.array(feat)


def extract_features(windows, fs=100, include_freq=True):
    """
    Extract features from all ECG windows.
    
    This function processes all signals and creates a feature matrix
    that can be used by traditional ML models.
    """
    n_windows = windows.shape[0]
    features = []
    
    print("Extracting features...")
    for i in tqdm(range(n_windows)):
        window = windows[i, :, 0]
        
        # Extract time-domain features
        time_feat = extract_time_features(window, fs)
        
        if include_freq:
            # Extract frequency-domain features
            freq_feat = extract_frequency_features(window, fs)
            # Combine both types of features
            feat = np.concatenate([time_feat, freq_feat])
        else:
            feat = time_feat
        
        features.append(feat)
    
    return np.stack(features, axis=0)

In [ ]:
# Extract features from all datasets
print("Extracting features for classical ML model...\n")
X_train_features = extract_features(X_train, fs=100)
X_val_features = extract_features(X_val, fs=100)
X_test_features = extract_features(X_test, fs=100)

print(f"\nFeature shapes:")
print(f"X_train_features: {X_train_features.shape} - {X_train_features.shape[1]} features per sample")
print(f"X_val_features:   {X_val_features.shape}")
print(f"X_test_features:  {X_test_features.shape}")

---
## Model 1: Simple Classification with Random Forest

### What is Random Forest?
Random Forest is an ensemble learning method that combines multiple decision trees:
- **Decision Tree**: Like a flowchart that makes decisions based on feature values
- **Random Forest**: Combines hundreds of trees to make more robust predictions
- **Voting**: Each tree votes, and the majority wins

### Why Random Forest?
- Works well with hand-crafted features
- Resistant to overfitting
- Can handle complex, non-linear relationships
- Provides feature importance rankings
- Fast to train and predict

### How It Works:
1. Create many random decision trees
2. Each tree trains on a random subset of data
3. Each tree uses a random subset of features
4. For prediction, all trees vote
5. The class with most votes wins

In [ ]:
print("="*60)
print("TRAINING MODEL 1: RANDOM FOREST CLASSIFIER")
print("="*60)

# Step 1: Feature scaling
# Random Forest doesn't strictly require scaling, but it can help
print("\nStep 1: Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_val_scaled = scaler.transform(X_val_features)
X_test_scaled = scaler.transform(X_test_features)
print("  - Features scaled using StandardScaler (mean=0, std=1)")

# Step 2: Initialize Random Forest
print("\nStep 2: Initializing Random Forest...")
print("  - n_estimators=200 (number of trees)")
print("  - Using all CPU cores for parallel training")

rf_model = RandomForestClassifier(
    n_estimators=200,        # Number of trees in the forest
    max_depth=None,          # Trees can grow as deep as needed
    random_state=42,         # For reproducibility
    n_jobs=-1,               # Use all CPU cores
    verbose=1                # Show progress
)

# Step 3: Train the model
print("\nStep 3: Training Random Forest...")
print("  - This may take a few minutes...")
rf_model.fit(X_train_scaled, y_train)

print("\n" + "="*60)
print("RANDOM FOREST TRAINING COMPLETE!")
print("="*60)

In [ ]:
# Evaluate Random Forest on validation set
print("Evaluating Random Forest on validation set...\n")

# Get predictions
y_val_pred_rf = rf_model.predict(X_val_scaled)
y_val_pred_proba_rf = rf_model.predict_proba(X_val_scaled)

# Calculate validation accuracy
val_acc = rf_model.score(X_val_scaled, y_val)
print(f"Validation Accuracy: {val_acc:.4f}")

# Calculate AUC for each class
print("\nValidation AUC per class:")
for i, label in enumerate(label_columns):
    # predict_proba returns list of arrays for multi-label
    y_score = y_val_pred_proba_rf[i][:, 1] if hasattr(y_val_pred_proba_rf[i], 'shape') else y_val_pred_proba_rf[i]
    auc = roc_auc_score(y_val[:, i], y_score)
    print(f"  {label}: {auc:.4f}")

---
## Model 2: Convolutional Neural Network (CNN)

### What is a CNN?
A Convolutional Neural Network is a deep learning model designed to automatically learn patterns from data:
- **No manual feature extraction needed** - the model learns features itself
- **Convolutional layers** act as automatic pattern detectors
- **Each layer learns progressively more complex patterns**

### CNN Architecture Components:

#### 1. **Convolutional Layers (Conv1D)**:
- Slide filters across the signal to detect patterns
- First layer: detects simple patterns (slopes, peaks)
- Deeper layers: combine simple patterns into complex features (QRS complex, P waves)
- **kernel_size**: how wide the pattern detector is

#### 2. **Batch Normalization**:
- Stabilizes training by normalizing layer outputs
- Speeds up convergence
- Reduces internal covariate shift

#### 3. **MaxPooling**:
- Downsamples the signal
- Keeps only the strongest activations
- Makes model more robust to small shifts in patterns
- Reduces computational cost

#### 4. **Global Average Pooling**:
- Converts variable-length feature maps to fixed-size vector
- Reduces overfitting compared to flattening

#### 5. **Dense Layers**:
- Learn complex combinations of extracted features
- Make final classification decisions

#### 6. **Dropout**:
- Randomly "drops" neurons during training
- Prevents overfitting by forcing model to learn redundant representations

#### 7. **Sigmoid Output**:
- Outputs probability for each class independently
- Perfect for multi-label classification (multiple diseases can be present)

In [ ]:
def build_cnn_model(input_shape, num_classes, dropout_rate=0.5):
    """
    Build a 1D CNN for ECG classification.
    
    Architecture:
    Input → Conv1D → BatchNorm → MaxPool
          → Conv1D → BatchNorm → MaxPool  
          → Conv1D → BatchNorm → GlobalAvgPool
          → Dense → Dropout → Dense (output)
    
    Args:
        input_shape: (time_steps, features) = (1000, 1) for our data
        num_classes: Number of output classes (5 in our case)
        dropout_rate: Dropout probability for regularization
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # First convolutional block
        # 32 filters, kernel size 7: learns 32 different 7-point patterns
        layers.Conv1D(32, kernel_size=7, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),  # Normalize activations
        layers.MaxPooling1D(pool_size=2),  # Reduce size by half
        
        # Second convolutional block  
        # 64 filters, kernel size 5: learns more complex patterns
        layers.Conv1D(64, kernel_size=5, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        
        # Third convolutional block
        # 128 filters, kernel size 3: learns even more complex patterns
        layers.Conv1D(128, kernel_size=3, activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),  # Convert to fixed-size vector
        
        # Dense classification head
        layers.Dense(128, activation='relu'),  # Learn combinations of features
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),  # Prevent overfitting
        
        # Output layer: one neuron per class with sigmoid activation
        layers.Dense(num_classes, activation='sigmoid')
    ])
    
    # Compile model
    model.compile(
        optimizer='adam',  # Adaptive learning rate optimizer
        loss='binary_crossentropy',  # Good for multi-label classification
        metrics=[
            'binary_accuracy',  # Overall accuracy
            tf.keras.metrics.AUC(name='auc', multi_label=True)  # Area under ROC curve
        ]
    )
    
    return model

In [ ]:
print("="*60)
print("TRAINING MODEL 2: CONVOLUTIONAL NEURAL NETWORK (CNN)")
print("="*60)

# Build the CNN model
print("\nBuilding CNN architecture...")
cnn_model = build_cnn_model(
    input_shape=(X_train.shape[1], X_train.shape[2]),  # (1000, 1)
    num_classes=num_classes,
    dropout_rate=0.5
)

# Display model architecture
print("\nModel Architecture:")
cnn_model.summary()

# Setup callbacks for training
print("\nSetting up training callbacks...")
callbacks_cnn = [
    # Stop training if validation loss doesn't improve for 10 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,  # Keep best model
        verbose=1
    ),
    # Reduce learning rate if validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,  # Multiply learning rate by 0.5
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

print("\nTraining CNN...")
print("  - Batch size: 32")
print("  - Max epochs: 50")
print("  - Early stopping patience: 10")
print("\nThis will take several minutes. Progress shown below:\n")

# Train the model
history_cnn = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks_cnn,
    verbose=1
)

print("\n" + "="*60)
print("CNN TRAINING COMPLETE!")
print("="*60)

In [ ]:
# Plot CNN training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot loss
axes[0].plot(history_cnn.history['loss'], label='Training Loss')
axes[0].plot(history_cnn.history['val_loss'], label='Validation Loss')
axes[0].set_title('CNN Model Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot AUC
axes[1].plot(history_cnn.history['auc'], label='Training AUC')
axes[1].plot(history_cnn.history['val_auc'], label='Validation AUC')
axes[1].set_title('CNN Model AUC', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nWhat these plots show:")
print("- Left: Loss decreases as model learns (lower is better)")
print("- Right: AUC increases as model improves (higher is better)")
print("- Gap between train and validation indicates overfitting")

---
## Model 3: LSTM (Long Short-Term Memory)

### What is LSTM?
LSTM is a type of Recurrent Neural Network (RNN) designed for sequential data:
- **Recurrent**: Processes data point by point, maintaining memory of previous points
- **Long Short-Term Memory**: Can remember patterns over long sequences
- **Perfect for time-series**: ECG signals are time-series data

### How LSTM Works:
Think of LSTM as reading the ECG signal from left to right, remembering important patterns:
1. **Forget Gate**: Decides what to forget from previous time steps
2. **Input Gate**: Decides what new information to store
3. **Output Gate**: Decides what to output based on current input and memory

### Why LSTM for ECG?
- ECG patterns have temporal dependencies (what happens now affects what comes next)
- Can capture long-range patterns (e.g., abnormal rhythm patterns)
- Naturally handles sequential data
- Can model the heart's rhythm dynamics

### Our Architecture:
We combine CNN with LSTM to get the best of both:
- **CNN layers**: Extract local patterns (like QRS complexes)
- **LSTM layers**: Model temporal dependencies and rhythm
- **Result**: Hybrid model that's more powerful than either alone

In [ ]:
def build_cnn_lstm_model(input_shape, num_classes, dropout_rate=0.5, lstm_units=64):
    """
    Build a hybrid CNN-LSTM model for ECG classification.
    
    Architecture:
    Input → Conv1D layers (feature extraction)
          → LSTM layers (temporal modeling)
          → Dense layers (classification)
    
    Why this hybrid approach?
    - CNN: Excellent at finding local patterns (QRS, P waves, T waves)
    - LSTM: Excellent at modeling temporal dependencies (rhythm, sequences)
    - Together: Capture both local features AND temporal patterns
    
    Args:
        input_shape: (time_steps, features)
        num_classes: Number of output classes
        dropout_rate: Dropout probability
        lstm_units: Number of LSTM units
    
    Returns:
        Compiled Keras model
    """
    inputs = layers.Input(shape=input_shape)
    
    # Convolutional feature extractor
    # These layers learn to detect local ECG patterns
    x = layers.Conv1D(32, kernel_size=7, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    x = layers.Conv1D(64, kernel_size=5, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    x = layers.Conv1D(128, kernel_size=3, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # Bidirectional LSTM layer
    # Processes sequence in both forward and backward directions
    # This helps capture both past and future context
    x = layers.Bidirectional(
        layers.LSTM(
            lstm_units,
            return_sequences=False,  # Only return final output
            dropout=0.2  # Dropout for LSTM regularization
        )
    )(x)
    
    # Dense classification head
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    
    # Output layer
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='CNN_LSTM')
    
    # Compile model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            'binary_accuracy',
            tf.keras.metrics.AUC(name='auc', multi_label=True)
        ]
    )
    
    return model

In [ ]:
print("="*60)
print("TRAINING MODEL 3: CNN-LSTM HYBRID")
print("="*60)

# Build the CNN-LSTM model
print("\nBuilding CNN-LSTM architecture...")
lstm_model = build_cnn_lstm_model(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    num_classes=num_classes,
    dropout_rate=0.5,
    lstm_units=64
)

# Display model architecture
print("\nModel Architecture:")
lstm_model.summary()

# Setup callbacks
print("\nSetting up training callbacks...")
callbacks_lstm = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

print("\nTraining CNN-LSTM...")
print("  - Batch size: 32")
print("  - Max epochs: 50")
print("  - Early stopping patience: 10")
print("\nNote: LSTM training is slower than CNN due to sequential processing.")
print("This will take several minutes. Progress shown below:\n")

# Train the model
history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks_lstm,
    verbose=1
)

print("\n" + "="*60)
print("CNN-LSTM TRAINING COMPLETE!")
print("="*60)

In [ ]:
# Plot LSTM training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot loss
axes[0].plot(history_lstm.history['loss'], label='Training Loss')
axes[0].plot(history_lstm.history['val_loss'], label='Validation Loss')
axes[0].set_title('CNN-LSTM Model Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot AUC
axes[1].plot(history_lstm.history['auc'], label='Training AUC')
axes[1].plot(history_lstm.history['val_auc'], label='Validation AUC')
axes[1].set_title('CNN-LSTM Model AUC', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nCompare with CNN training curves to see performance differences.")

---
## Step 7: Model Comparison and Evaluation

Now we'll evaluate all three models on the test set and compare their performance.

### Evaluation Metrics:

#### 1. **Accuracy**:
- Percentage of correct predictions
- Simple but can be misleading with imbalanced classes

#### 2. **AUC (Area Under ROC Curve)**:
- Measures model's ability to distinguish between classes
- Range: 0.5 (random) to 1.0 (perfect)
- 0.7-0.8: Acceptable
- 0.8-0.9: Excellent
- 0.9+: Outstanding

#### 3. **Precision**:
- Of all positive predictions, how many were correct?
- Important when false positives are costly

#### 4. **Recall (Sensitivity)**:
- Of all actual positives, how many did we find?
- Important in medical diagnosis (don't miss diseases)

#### 5. **F1-Score**:
- Harmonic mean of precision and recall
- Balances both metrics

In [ ]:
print("="*80)
print("FINAL MODEL EVALUATION ON TEST SET")
print("="*80)

# Dictionary to store results
results = {}

# 1. Evaluate Random Forest
print("\n" + "="*80)
print("MODEL 1: RANDOM FOREST")
print("="*80)

y_test_pred_rf = rf_model.predict(X_test_scaled)
y_test_pred_proba_rf = rf_model.predict_proba(X_test_scaled)

# Calculate overall accuracy
rf_accuracy = rf_model.score(X_test_scaled, y_test)
print(f"\nTest Accuracy: {rf_accuracy:.4f}")

# Calculate per-class AUC
rf_auc_scores = []
print("\nPer-Class AUC Scores:")
for i, label in enumerate(label_columns):
    y_score = y_test_pred_proba_rf[i][:, 1]
    auc = roc_auc_score(y_test[:, i], y_score)
    rf_auc_scores.append(auc)
    print(f"  {label:8s}: {auc:.4f}")

rf_macro_auc = np.mean(rf_auc_scores)
print(f"\nMacro-Average AUC: {rf_macro_auc:.4f}")

results['Random Forest'] = {
    'accuracy': rf_accuracy,
    'macro_auc': rf_macro_auc,
    'class_auc': rf_auc_scores
}

# 2. Evaluate CNN
print("\n" + "="*80)
print("MODEL 2: CONVOLUTIONAL NEURAL NETWORK (CNN)")
print("="*80)

y_test_pred_cnn = cnn_model.predict(X_test)
test_results_cnn = cnn_model.evaluate(X_test, y_test, verbose=0)

cnn_accuracy = test_results_cnn[1]  # binary_accuracy
cnn_auc = test_results_cnn[2]  # auc

print(f"\nTest Accuracy: {cnn_accuracy:.4f}")
print(f"Test AUC: {cnn_auc:.4f}")

# Calculate per-class AUC
cnn_auc_scores = []
print("\nPer-Class AUC Scores:")
for i, label in enumerate(label_columns):
    auc = roc_auc_score(y_test[:, i], y_test_pred_cnn[:, i])
    cnn_auc_scores.append(auc)
    print(f"  {label:8s}: {auc:.4f}")

cnn_macro_auc = np.mean(cnn_auc_scores)
print(f"\nMacro-Average AUC: {cnn_macro_auc:.4f}")

results['CNN'] = {
    'accuracy': cnn_accuracy,
    'macro_auc': cnn_macro_auc,
    'class_auc': cnn_auc_scores
}

# 3. Evaluate CNN-LSTM
print("\n" + "="*80)
print("MODEL 3: CNN-LSTM HYBRID")
print("="*80)

y_test_pred_lstm = lstm_model.predict(X_test)
test_results_lstm = lstm_model.evaluate(X_test, y_test, verbose=0)

lstm_accuracy = test_results_lstm[1]  # binary_accuracy
lstm_auc = test_results_lstm[2]  # auc

print(f"\nTest Accuracy: {lstm_accuracy:.4f}")
print(f"Test AUC: {lstm_auc:.4f}")

# Calculate per-class AUC
lstm_auc_scores = []
print("\nPer-Class AUC Scores:")
for i, label in enumerate(label_columns):
    auc = roc_auc_score(y_test[:, i], y_test_pred_lstm[:, i])
    lstm_auc_scores.append(auc)
    print(f"  {label:8s}: {auc:.4f}")

lstm_macro_auc = np.mean(lstm_auc_scores)
print(f"\nMacro-Average AUC: {lstm_macro_auc:.4f}")

results['CNN-LSTM'] = {
    'accuracy': lstm_accuracy,
    'macro_auc': lstm_macro_auc,
    'class_auc': lstm_auc_scores
}

---
## Step 8: Comprehensive Model Comparison

Let's visualize and compare all three models side by side.

In [ ]:
# Create comparison table
print("="*80)
print("OVERALL MODEL COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'CNN', 'CNN-LSTM'],
    'Test Accuracy': [
        results['Random Forest']['accuracy'],
        results['CNN']['accuracy'],
        results['CNN-LSTM']['accuracy']
    ],
    'Macro AUC': [
        results['Random Forest']['macro_auc'],
        results['CNN']['macro_auc'],
        results['CNN-LSTM']['macro_auc']
    ]
})

print("\n", comparison_df.to_string(index=False))

# Find best model
best_model_idx = comparison_df['Macro AUC'].argmax()
best_model = comparison_df.loc[best_model_idx, 'Model']
best_auc = comparison_df.loc[best_model_idx, 'Macro AUC']

print(f"\n🏆 Best Model: {best_model} (AUC: {best_auc:.4f})")
print("="*80)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Overall metrics comparison
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(x - width/2, comparison_df['Test Accuracy'], width, label='Accuracy', alpha=0.8)
axes[0].bar(x + width/2, comparison_df['Macro AUC'], width, label='Macro AUC', alpha=0.8)
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Model'])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, 1])

# Plot 2: Per-class AUC comparison
x_classes = np.arange(len(label_columns))
width = 0.25

axes[1].bar(x_classes - width, results['Random Forest']['class_auc'], width, 
            label='Random Forest', alpha=0.8)
axes[1].bar(x_classes, results['CNN']['class_auc'], width, 
            label='CNN', alpha=0.8)
axes[1].bar(x_classes + width, results['CNN-LSTM']['class_auc'], width, 
            label='CNN-LSTM', alpha=0.8)

axes[1].set_xlabel('Disease Class', fontsize=12)
axes[1].set_ylabel('AUC Score', fontsize=12)
axes[1].set_title('Per-Class AUC Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_classes)
axes[1].set_xticklabels(label_columns, rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
# Create detailed per-class comparison table
print("\n" + "="*80)
print("PER-CLASS AUC COMPARISON")
print("="*80)

per_class_df = pd.DataFrame({
    'Class': label_columns,
    'Random Forest': results['Random Forest']['class_auc'],
    'CNN': results['CNN']['class_auc'],
    'CNN-LSTM': results['CNN-LSTM']['class_auc']
})

# Add a column showing which model is best for each class
per_class_df['Best Model'] = per_class_df[['Random Forest', 'CNN', 'CNN-LSTM']].idxmax(axis=1)

print("\n", per_class_df.to_string(index=False))
print("="*80)

---
## Step 9: Key Insights and Interpretations

### Understanding the Results:

#### 1. **Random Forest (Traditional ML)**:
**Strengths:**
- Fast to train
- Interpretable (can see feature importance)
- Works well with small datasets
- No need for GPU

**Limitations:**
- Requires manual feature engineering
- May miss complex patterns
- Performance ceiling limited by feature quality

#### 2. **CNN (Deep Learning)**:
**Strengths:**
- Learns features automatically
- Excellent at detecting local patterns
- Faster than LSTM
- Parallelizable (GPU-friendly)

**Limitations:**
- Requires more data
- Less interpretable
- May miss long-range temporal dependencies

#### 3. **CNN-LSTM (Hybrid Deep Learning)**:
**Strengths:**
- Combines CNN pattern detection with LSTM temporal modeling
- Captures both local and global patterns
- Best for complex temporal patterns
- Usually achieves highest performance

**Limitations:**
- Slowest to train
- Most complex architecture
- Requires most data
- Higher computational requirements

### When to Use Each Model:

- **Use Random Forest when**:
  - You have limited data
  - You need interpretability
  - You want quick results
  - You don't have GPU resources

- **Use CNN when**:
  - You have sufficient data
  - You need fast inference
  - Patterns are mostly local
  - You have GPU resources

- **Use CNN-LSTM when**:
  - You have lots of data
  - Temporal dependencies are important
  - You need best possible performance
  - Training time is not critical

---
## Step 10: Conclusion and Next Steps

### What We Accomplished:

1. ✅ Downloaded and preprocessed the PTB-XL ECG dataset
2. ✅ Extracted meaningful features from ECG signals
3. ✅ Built and trained three different types of models:
   - Random Forest (traditional ML)
   - CNN (deep learning)
   - CNN-LSTM (hybrid deep learning)
4. ✅ Evaluated and compared model performance
5. ✅ Visualized results and gained insights

### Key Takeaways:

- **Data preprocessing is crucial**: Proper normalization and denoising significantly impact results
- **Different approaches have different strengths**: No single model is always best
- **Deep learning excels with raw data**: CNNs and LSTMs automatically learn relevant features
- **Hybrid models often perform best**: Combining CNN and LSTM captures both local and temporal patterns
- **Evaluation metrics matter**: Use multiple metrics (accuracy, AUC, precision, recall) for medical applications

### Possible Improvements:

1. **Data Augmentation**: Add noise, time-warping, or amplitude scaling to increase training data
2. **Use All 12 Leads**: We only used Lead II; using all leads could improve performance
3. **Attention Mechanisms**: Add attention layers to focus on important signal regions
4. **Ensemble Methods**: Combine predictions from multiple models
5. **Transfer Learning**: Use pre-trained models from similar tasks
6. **Hyperparameter Tuning**: Optimize learning rates, layer sizes, etc.
7. **Class Balancing**: Address class imbalance with weighted losses or sampling

### Further Learning:

- Study ECG physiology to understand what patterns indicate disease
- Explore attention mechanisms and explainable AI
- Learn about medical ML best practices and regulations
- Experiment with other architectures (Transformers, ResNets, etc.)

### Important Note:

⚠️ **These models are for educational purposes only. They should never be used for actual medical diagnosis without:**
- Extensive validation on diverse populations
- Clinical trials and regulatory approval
- Integration with clinical decision support systems
- Oversight by qualified medical professionals

---

**Congratulations! You've completed a comprehensive tutorial on cardiovascular disease detection using machine learning!** 🎉

---
## Appendix: Save and Load Models

Here's how to save your trained models for later use.

In [ ]:
import pickle

# Create models directory
os.makedirs('saved_models', exist_ok=True)

# Save Random Forest model
with open('saved_models/random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
with open('saved_models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save CNN model
cnn_model.save('saved_models/cnn_model.h5')

# Save CNN-LSTM model
lstm_model.save('saved_models/cnn_lstm_model.h5')

print("All models saved successfully!")
print("\nTo load models later:")
print("  Random Forest: pickle.load(open('saved_models/random_forest_model.pkl', 'rb'))")
print("  CNN: tf.keras.models.load_model('saved_models/cnn_model.h5')")
print("  CNN-LSTM: tf.keras.models.load_model('saved_models/cnn_lstm_model.h5')")